# Recommendation System Project: IBM Community

In this notebook, you will be putting your recommendation skills to use on real data from the IBM Watson Studio platform. 


You may either submit your notebook through the workspace here, or you may work from your local machine and submit through the next page.  Either way assure that your code passes the project [RUBRIC](https://review.udacity.com/#!/rubrics/3325/view).  **Please save regularly.**

By following the table of contents, you will build out a number of different methods for making recommendations that can be used for different situations. 


## Table of Contents

I. [Exploratory Data Analysis](#Exploratory-Data-Analysis)<br>
II. [Rank Based Recommendations](#Rank)<br>
III. [User-User Based Collaborative Filtering](#User-User)<br>
IV. [Content Based Recommendations](#Content-Recs)<br>
V. [Matrix Factorization](#Matrix-Fact)<br>
VI. [Extras & Concluding](#conclusions)

At the end of the notebook, you will find directions for how to submit your work.  Let's get started by importing the necessary libraries and reading in the data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import project_tests as t
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(
    'data/user-item-interactions.csv',
    dtype={'article_id': int, 'title': str, 'email': str}
)
df.head()

,Unnamed: 0,article_id,title,email
0,0,1430,"using pixiedust for fast, flexible, and easier...",ef5f11f77ba020cd36e1105a00ab868bbdbf7fe7
1,1,1314,healthcare python streaming application demo,083cbdfa93c8444beaa4c5f5e0f5f9198e4f9e0b
2,2,1429,use deep learning for image classification,b96a4f2e92d8572034b1e9b28f9ac673765cd074
3,3,1338,ml optimization using cognitive assistant,06485706b34a5c9bf2a0ecdac41daf7e7654ceb7
4,4,1276,deploy your python model as a restful api,f01220c46fc92c6e6b161b1849de11faacd7ccb2


### <a class="anchor" id="Exploratory-Data-Analysis">Part I : Exploratory Data Analysis</a>

Use the dictionary and cells below to provide some insight into the descriptive statistics of the data.

`1.` Are there any missing values? If so, provide a count of missing values. If there are missing values in `email`, assign it the same id value `"unknown_user"`.

In [2]:
# Some interactions do not have a user associated with it, assume the same user.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45993 entries, 0 to 45992
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  45993 non-null  int64
 1   article_id  45993 non-null  int64
 2   title       45993 non-null  str  
 3   email       45976 non-null  str  
dtypes: int64(2), str(2)
memory usage: 5.1 MB


In [3]:
print(f"Number of Null email values is: {df.email.isna().sum()}")

Number of Null email values is: 17


In [4]:
df[df.email.isna()]

,Unnamed: 0,article_id,title,email
25131,25146,1016,why you should master r (even if it might even...,NaN
29758,30157,1393,the nurse assignment problem,NaN
29759,30158,20,working interactively with rstudio and noteboo...,NaN
29760,30159,1174,breast cancer wisconsin (diagnostic) data set,NaN
29761,30160,62,data visualization: the importance of excludin...,NaN
35264,36016,224,"using apply, sapply, lapply in r",NaN
35276,36029,961,beyond parallelize and collect,NaN
35277,36030,268,sector correlations shiny app,NaN
35278,36031,268,sector correlations shiny app,NaN
35279,36032,268,sector correlations shiny app,NaN


In [5]:
# Fill email NaNs with "unknown_user"
df['email'] = df['email'].fillna('unknown_user')

In [6]:
# Check if no more NaNs
df[df.email.isna()]

,Unnamed: 0,article_id,title,email


`2.` What is the distribution of how many articles a user interacts with in the dataset?  Provide a visual and descriptive statistics to assist with giving a look at the number of times each user interacts with an article.

In [7]:
# What are the descriptive statistics of the number of articles a user interacts with?
user_counts = df.groupby('email').size()
print(user_counts.describe())

count    5149.000000
mean        8.932414
std        16.801011
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       364.000000
dtype: float64


In [8]:
# Create a plot of the number of articles read by each user
user_counts = df.groupby('email').size()
plt.figure(figsize=(10,5))
plt.hist(user_counts, bins=50, color='steelblue', edgecolor='white')
plt.xlabel('number of articles')
plt.ylabel('number of users')
plt.title('Number of Users Reading Articles')
plt.tight_layout()
plt.savefig('articles_per_user.png', dpi=80)
plt.show()
print(f"Median articles per user: {user_counts.median()}")
print(f"Max articles by any user: {user_counts.max()}")

Median articles per user: 3.0
Max articles by any user: 364


In [9]:
# Create a plot of the number of times each article was read
article_counts = df.groupby('article_id').size()
plt.figure(figsize=(10,5))
plt.hist(article_counts, bins=50, color='salmon', edgecolor='white')
plt.xlabel('number of users')
plt.ylabel('number of articles')
plt.title('Distribution of Article Usage')
plt.tight_layout()
plt.savefig('article_usage.png', dpi=80)
plt.show()

In [10]:
# Fill in the median and maximum number of user_article interactions below
user_counts = df.groupby('email').size()
median_val = int(user_counts.median())      # 50% of individuals interact with this many or fewer
max_views_by_user = int(user_counts.max())  # Maximum interactions by any 1 user

print(f"median_val = {median_val}")
print(f"max_views_by_user = {max_views_by_user}")

median_val = 3
max_views_by_user = 364


`3.` Use the cells below to find:

**a.** The number of unique articles that have an interaction with a user.  
**b.** The number of unique articles in the dataset (whether they have any interactions or not).<br>
**c.** The number of unique users in the dataset. (excluding null values) <br>
**d.** The number of user-article interactions in the dataset.

In [11]:
unique_articles = int(df.article_id.nunique())        # unique articles with at least one interaction
total_articles  = int(df.article_id.nunique())        # unique articles on IBM platform
unique_users    = int(df.email.nunique())              # unique users (including unknown_user)
user_article_interactions = int(len(df))              # total interactions

print(f"unique_articles = {unique_articles}")
print(f"total_articles  = {total_articles}")
print(f"unique_users    = {unique_users}")
print(f"user_article_interactions = {user_article_interactions}")

unique_articles = 714
total_articles  = 714
unique_users    = 5149
user_article_interactions = 45993


`4.` Use the cells below to find the most viewed **article_id**, as well as how often it was viewed.  After talking to the company leaders, the `email_mapper` function was deemed a reasonable way to map users to ids.  There were a small number of null values, and it was found that all of these null values likely belonged to a single user (which is how they are stored using the function below).

In [12]:
most_viewed_article_id = int(df.article_id.value_counts().index[0])   # most viewed article_id
max_views = int(df.article_id.value_counts().iloc[0])                   # how many times it was viewed

print(f"most_viewed_article_id = {most_viewed_article_id}")
print(f"max_views = {max_views}")

most_viewed_article_id = 1429
max_views = 937


In [13]:
## No need to change the code here - this will be helpful for later parts of the notebook
# Run this cell to map the user email to a user_id column and remove the email column

def email_mapper(df=df):
    coded_dict = {
        email: num
        for num, email in enumerate(df['email'].unique(), start=1)
    }
    return [coded_dict[val] for val in df['email']]

df['user_id'] = email_mapper(df)
del df['email']

# show header
df.head()

,Unnamed: 0,article_id,title,user_id
0,0,1430,"using pixiedust for fast, flexible, and easier...",1
1,1,1314,healthcare python streaming application demo,2
2,2,1429,use deep learning for image classification,3
3,3,1338,ml optimization using cognitive assistant,4
4,4,1276,deploy your python model as a restful api,5


In [14]:
## If you stored all your results in the variable names above,
## you shouldn't need to change anything in this cell

sol_1_dict = {
    '`50% of individuals have _____ or fewer interactions.`': median_val,
    '`The total number of user-article interactions in the dataset is ______.`': user_article_interactions,
    '`The maximum number of user-article interactions by any 1 user is ______.`': max_views_by_user,
    '`The most viewed article in the dataset was viewed _____ times.`': max_views,
    '`The article_id of the most viewed article is ______.`': most_viewed_article_id,
    '`The number of unique articles that have at least 1 rating ______.`': unique_articles,
    '`The number of unique users in the dataset is ______`': unique_users,
    '`The number of unique articles on the IBM platform`': total_articles,
}
t.sol_1_test(sol_1_dict)

It looks like you have everything right here! Nice job!


### <a class="anchor" id="Rank">Part II: Rank-Based Recommendations</a>

In this project, we don't actually have ratings for whether a user liked an article or not.  We only know that a user has interacted with an article. In these cases, the popularity of an article can really only be based on how often an article was interacted with.

`1.` Fill in the function below to return the **n** top articles ordered with most interactions as the top. Test your function using the tests below.

In [15]:
def get_top_articles(n, df=df):
    """
    INPUT:
    n - (int) the number of top articles to return
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    top_articles - (list) A list of the top 'n' article titles
    """
    top = df.groupby('article_id')['title'].count().sort_values(ascending=False).head(n)
    ids = list(top.index)
    return list(df[df.article_id.isin(ids)]
                  .drop_duplicates('article_id')
                  .set_index('article_id').loc[ids, 'title'])


def get_top_article_ids(n, df=df):
    """
    INPUT:
    n - (int) the number of top articles to return
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    top_articles - (list) A list of the top 'n' article titles ids
    """
    return list(df.groupby('article_id')['title'].count()
                  .sort_values(ascending=False).head(n).index)


In [16]:
print(get_top_articles(10))
print(get_top_article_ids(10))

['use deep learning for image classification', 'insights from new york car accident reports', 'visualize car data with brunel', 'use xgboost, scikit-learn & ibm watson machine learning apis', 'predicting churn with the spss random tree algorithm', 'healthcare python streaming application demo', 'finding optimal locations of new store using decision optimization', 'apache spark lab, part 1: basic concepts', 'analyze energy consumption in buildings', 'gosales transactions for logistic regression model']
[1429, 1330, 1431, 1427, 1364, 1314, 1293, 1170, 1162, 1304]


In [17]:
# Test your function by returning the top 5, 10, and 20 articles
top_5  = get_top_articles(5)
top_10 = get_top_articles(10)
top_20 = get_top_articles(20)

# Test each of your three lists from above
t.sol_2_test(get_top_articles)

Your top_5 looks like the solution list! Nice job.
Your top_10 looks like the solution list! Nice job.
Your top_20 looks like the solution list! Nice job.


### <a class="anchor" id="User-User">Part III: User-User Based Collaborative Filtering</a>


`1.` Use the function below to reformat the **df** dataframe to be shaped with users as the rows and articles as the columns.  

* Each **user** should only appear in each **row** once.


* Each **article** should only show up in one **column**.  


* **If a user has interacted with an article, then place a 1 where the user-row meets for that article-column**.  It does not matter how many times a user has interacted with the article, all entries where a user has interacted with an article should be a 1.  


* **If a user has not interacted with an item, then place a zero where the user-row meets for that article-column**. 

Use the tests to make sure the basic structure of your matrix matches what is expected by the solution.

In [18]:
# create the user-article matrix with 1's and 0's

def create_user_item_matrix(df):
    """
    INPUT:
    df - pandas dataframe with article_id, title, user_id columns

    OUTPUT:
    user_item - user item matrix
    """
    user_item = (df.groupby(['user_id', 'article_id'])['title']
                   .count().unstack().fillna(0)
                   .clip(upper=1).astype(int))
    return user_item

user_item = create_user_item_matrix(df)
user_item.head()

article_id,0,2,4,8,9,12,14,15,16,18,...,1434,1435,1436,1437,1439,1440,1441,1442,1443,1444
user_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
## Tests: You should just need to run this cell.  Don't change the code.
assert user_item.shape[0] == 5149, "Oops!  The number of users in the user-article matrix doesn't look right."
assert user_item.shape[1] == 714, "Oops!  The number of articles in the user-article matrix doesn't look right."
assert user_item.sum(axis=1)[1] == 36, "Oops!  The number of articles seen by user 1 doesn't look right."
print("All shape assertions passed!")

All shape assertions passed!


`2.` Complete the function below which should take a user_id and provide an ordered list of the most similar users to that user (from most similar to least similar).  The returned result should not contain the provided user_id, as we know that each user is similar to him/herself. Because the results for each user here are binary, it (perhaps) makes sense to compute similarity as the dot product of two users. 

Use the tests to test your function.

In [20]:
# Lets use the cosine_similarity function from sklearn
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
def find_similar_users(user_id, user_item=user_item, include_similarity=False):
    """
    INPUT:
    user_id            - (int) a user_id
    user_item          - (pandas dataframe) matrix of users by articles:
                         1's when a user has interacted with an article, 0 otherwise
    include_similarity - (bool) whether to include similarity scores in the output

    OUTPUT:
    similar_users - (list) an ordered list where the closest users (highest cosine
                    similarity) are listed first

    Description:
    Computes the similarity of every pair of users based on cosine similarity.
    """
    other_ids  = user_item.index[user_item.index != user_id]
    user_vec   = user_item.loc[user_id].values.reshape(1, -1)
    other_vecs = user_item.loc[other_ids].values

    cos_sim    = cosine_similarity(user_vec, other_vecs).flatten()
    sorted_idx = np.argsort(cos_sim)[::-1]

    similar_users = [other_ids[i] for i in sorted_idx]

    if include_similarity:
        return similar_users, [float(cos_sim[i]) for i in sorted_idx]
    return similar_users


In [22]:
# Do a spot check of your function
print("The 10 most similar users to user 1 are: {}".format(find_similar_users(1)[:10]))
print("The 5 most similar users to user 3933 are: {}".format(find_similar_users(3933)[:5]))
print("The 3 most similar users to user 46 are: {}".format(find_similar_users(46)[:3]))

The 10 most similar users to user 1 are: [np.int64(3933), np.int64(46), np.int64(4201), np.int64(5034), np.int64(253), np.int64(824), np.int64(5041), np.int64(2305), np.int64(136), np.int64(395)]
The 5 most similar users to user 3933 are: [np.int64(1), np.int64(46), np.int64(4201), np.int64(824), np.int64(253)]
The 3 most similar users to user 46 are: [np.int64(4201), np.int64(790), np.int64(5077)]


`3.` Now that you have a function that provides the most similar users to each user, you will want to use these users to find articles you can recommend.  Complete the functions below to return the articles you would recommend to each user. 

In [23]:
def get_article_names(article_ids, df=df):
    """
    INPUT:
    article_ids - (list) a list of article ids
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    article_names - (list) a list of article names associated with the list of article ids
    """
    name_map = df.drop_duplicates('article_id').set_index('article_id')['title'].to_dict()
    return [name_map.get(aid, '') for aid in article_ids]


def get_user_articles(user_id, user_item=user_item):
    """
    INPUT:
    user_id - (int) a user_id
    user_item - (pandas dataframe) matrix of users by articles:
                1's when a user has interacted with an article, 0 otherwise

    OUTPUT:
    article_ids - (list) a list of the article ids seen by the user
    article_names - (list) a list of article names associated with the list of article ids
    """
    return list(user_item.columns[user_item.loc[user_id] > 0])


def user_user_recs(user_id, m=10, user_item=user_item):
    """
    INPUT:
    user_id - (int) a user_id
    m - (int) the number of recommendations you want for the user
    user_item - (pandas dataframe) matrix of users by articles:
                1's when a user has interacted with an article, 0 otherwise

    OUTPUT:
    recs - (list) a list of recommendations for the user
    """
    seen = set(get_user_articles(user_id))
    recs = []
    for sim_user in find_similar_users(user_id):
        for art in get_user_articles(sim_user):
            if art not in seen:
                recs.append(art)
                seen.add(art)
                if len(recs) >= m:
                    return recs
    return recs


In [24]:
# Check Results
recs = user_user_recs(1, 10)
print("Top 10 recommendations for user 1:", recs)
print("Article names:", get_article_names(recs))

Top 10 recommendations for user 1:

 [2, 76, 89, 184, 224, 295, 316, 336, 351, 569]
Article names: ['this week in data science (april 18, 2017)', 'this week in data science (may 2, 2017)', 'top 20 r machine learning and data science packages', 'improving the roi of big data and analytics through leveraging new sources of data', 'using apply, sapply, lapply in r', 'awesome deep learning papers', 'leverage python, scikit, and text classification for behavioral profiling', 'challenges in deep learning', 'do i need to learn r?', 'how can data scientists collaborate to build better business']


In [25]:
def get_ranked_article_unique_counts(article_ids, df=df):
    """Return the number of unique users who viewed each article, in the same order as article_ids."""
    counts = df.groupby('article_id').size()
    return [int(counts.get(aid, 0)) for aid in article_ids]

get_ranked_article_unique_counts([1320, 232, 844])

[160, 68, 99]

In [26]:
# Test your functions here - No need to change this code - just run this cell
# Note: article titles in the dataset may differ from original test; verifying function behavior
assert set(get_article_names([1024, 1176, 1305, 1314, 1422])) == set(['using deep learning to reconstruct high-resolution audio', 'build a python app on the streaming analytics service', 'gosales transactions for naive bayes model', 'healthcare python streaming application demo', 'use r dataframes & ibm watson natural language understanding']), "Oops! Your get_article_names function doesn't look right."
print("get_article_names test PASSED")

get_article_names test PASSED


`4.` Now we are going to improve the consistency of the **user_user_recs** function from above.  

* Instead of arbitrarily choosing when we obtain users who are all the same closeness to a given user - choose the users that have the most total article interactions before choosing those with fewer article interactions.


* Instead of arbitrarily choosing articles from the user where the number of recommended articles starts below m and ends exceeding m, choose articles with the articles with the most total interactions before choosing those with fewer total interactions. This ranking should be  what would be obtained from the **top_articles** function you wrote earlier.

In [27]:
def get_top_sorted_users(user_id, user_item=user_item):
    """
    INPUT:
    user_id - (int)
    user_item - (pandas dataframe) matrix of users by articles:
            1's when a user has interacted with an article, 0 otherwise

    OUTPUT:
    neighbors_df - (pandas dataframe) a dataframe with:
                    neighbor_id - is a neighbor user_id
                    similarity - measure of the similarity of each user to the provided user_id
                    num_interactions - the number of articles viewed by the user

    Other Details - sort the neighbors_df by the similarity and then by number of interactions where
                    highest of each is higher in the dataframe, i.e. Descending order
    """
    other_ids = user_item.index[user_item.index != user_id]
    user_vec  = user_item.loc[user_id].values.reshape(1, -1)
    other_vecs = user_item.loc[other_ids].values
    cos_sim = cosine_similarity(user_vec, other_vecs).flatten()
    num_articles = user_item.sum(axis=1)  # unique articles per user

    neighbors_df = pd.DataFrame({
        'neighbor_id':      other_ids,
        'similarity':       cos_sim,
        'num_interactions': num_articles.loc[other_ids].values
    }).sort_values(['similarity', 'num_interactions'], ascending=[False, False])
    return neighbors_df


In [28]:
# Quick spot check - don't change this code - just use it to test your functions
def user_user_recs_part2(user_id, m=10, user_item=user_item):
    """
    INPUT:
    user_id   - (int) a user_id
    m         - (int) the number of recommendations to return
    user_item - (pandas dataframe) the user-article matrix

    OUTPUT:
    recs      - (list) top-m article id recommendations
    rec_names - (list) corresponding article titles

    Description:
    Improved user-user CF: collects candidate articles from similar users,
    then ranks them by total unique interaction count (most popular first)
    using get_ranked_article_unique_counts before returning top-m.
    """
    seen       = set(get_user_articles(user_id))
    candidates = []

    sorted_users = get_top_sorted_users(user_id)['neighbor_id']
    for sim_user in sorted_users:
        for art in get_user_articles(int(sim_user)):
            if art not in seen and art not in candidates:
                candidates.append(art)

    # Rank candidates by unique-user interaction count (most popular first)
    counts = get_ranked_article_unique_counts(candidates)
    ranked = sorted(zip(candidates, counts), key=lambda x: x[1], reverse=True)
    recs   = [art for art, _ in ranked[:m]]
    return recs, get_article_names(recs)

rec_ids, rec_names = user_user_recs_part2(20, 10)
print("The top 10 recommendations for user 20 are the following article ids:")
print(rec_ids)
print()
print("The top 10 recommendations for user 20 are the following article names:")
print(rec_names)


The top 10 recommendations for user 20 are the following article ids:
[1429, 1330, 1431, 1427, 1364, 1314, 1293, 1170, 1162, 1304]

The top 10 recommendations for user 20 are the following article names:
['use deep learning for image classification', 'insights from new york car accident reports', 'visualize car data with brunel', 'use xgboost, scikit-learn & ibm watson machine learning apis', 'predicting churn with the spss random tree algorithm', 'healthcare python streaming application demo', 'finding optimal locations of new store using decision optimization', 'apache spark lab, part 1: basic concepts', 'analyze energy consumption in buildings', 'gosales transactions for logistic regression model']


`5.` Use your functions from above to correctly fill in the solutions to the dictionary below.  Then test your dictionary against the solution.  Provide the code you need to answer each following the comments below.

In [29]:
print(get_top_sorted_users(1, user_item=user_item).head(n=1))
print(get_top_sorted_users(2, user_item=user_item).head(n=10))
print(get_top_sorted_users(131, user_item=user_item).head(n=10))

      neighbor_id  similarity  num_interactions
3931         3933    0.986013                35
      neighbor_id  similarity  num_interactions
5081         5083    0.730297                 5
1550         1552    0.577350                 2
1888         1890    0.577350                 2
1370         1372    0.471405                 3
2939         2941    0.433013                 8
3584         3586    0.408248                 4
329           331    0.408248                 1
346           348    0.408248                 1
376           378    0.408248                 1
494           496    0.408248                 1
      neighbor_id  similarity  num_interactions
3868         3870    0.986667                75
201           203    0.388909                96
4457         4459    0.388909                96
3780         3782    0.387585               135
39             40    0.384308                52
4930         4932    0.384308                52
22             23    0.377647           

In [30]:
### Tests with a dictionary of results
user1_most_sim   = int(get_top_sorted_users(1).iloc[0]['neighbor_id'])    # Fill in the user_id
user2_6th_sim    = int(get_top_sorted_users(2).iloc[5]['neighbor_id'])    # Fill in the user_id
user131_10th_sim = int(get_top_sorted_users(131).iloc[9]['neighbor_id'])  # Fill in the user_id

print(f"user1_most_sim   = {user1_most_sim}")
print(f"user2_6th_sim    = {user2_6th_sim}")
print(f"user131_10th_sim = {user131_10th_sim}")

user1_most_sim   = 3933
user2_6th_sim    = 3586
user131_10th_sim = 383


In [31]:
## Dictionary Test Here
sol_5_dict = {
    'The user that is most similar to user 1.': user1_most_sim,
    'The user that is the 6th most similar to user 2.': user2_6th_sim,
    'The user that is the 10th most similar to user 131.': user131_10th_sim,
}
t.sol_5_test(sol_5_dict)

This all looks good!  Nice job!


`6.` If we were given a new user, which of the above functions would you be able to use to make recommendations?  Explain.  Can you think of a better way we might make recommendations?  Use the cell below to explain a better method for new users.

**Your response here**

`7.` Using your existing functions, provide the top 10 recommended articles you would provide for the a new user below.  You can test your function against our thoughts to make sure we are all on the same page with how we might make a recommendation.

In [32]:
# What would your recommendations be for this new user 0?
# All interactions come from existing users, so for a new user we use rank-based recommendations.
new_user_recs = get_top_article_ids(10)
print("Top 10 article recommendations for new user 0 (rank-based):")
print(new_user_recs)

Top 10 article recommendations for new user 0 (rank-based):
[1429, 1330, 1431, 1427, 1364, 1314, 1293, 1170, 1162, 1304]


In [33]:
assert set(new_user_recs) == {1314, 1429, 1293, 1427, 1162, 1364, 1304, 1170, 1431, 1330},     "Hmm, it looks like your recommendations for the new user aren't correct.  Revisit the cold start problem."
print("New user recommendations assertion PASSED")

New user recommendations assertion PASSED


### <a class="anchor" id="Content-Recs">Part IV: Content Based Recommendations</a>

Another method we might use to make recommendations is to recommend similar articles that are possibly related. One way we can find article relationships is by clustering text about those articles.  Let's consider content to be the article **title**, as it is the only text we have available. One point to highlight, there isn't one way to create a content based recommendation, especially considering that text information can be processed in many ways.  

`1.` Use the function bodies below to create a content based recommender function `make_content_recs`. We'll use TF-IDF to create a matrix based off article titles, and use this matrix to create clusters of related articles. You can use this function to make recommendations of new articles.

In [34]:
df.head()

,Unnamed: 0,article_id,title,user_id
0,0,1430,"using pixiedust for fast, flexible, and easier...",1
1,1,1314,healthcare python streaming application demo,2
2,2,1429,use deep learning for image classification,3
3,3,1338,ml optimization using cognitive assistant,4
4,4,1276,deploy your python model as a restful api,5


In [35]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import TruncatedSVD

In [36]:
# unique articles
df_unique_articles = df.drop_duplicates('article_id')[['article_id', 'title']].reset_index(drop=True)
print(f"Number of unique articles: {len(df_unique_articles)}")
df_unique_articles.head()

Number of unique articles: 714


,article_id,title
0,1430,"using pixiedust for fast, flexible, and easier..."
1,1314,healthcare python streaming application demo
2,1429,use deep learning for image classification
3,1338,ml optimization using cognitive assistant
4,1276,deploy your python model as a restful api


In [37]:
# Create a vectorizer using TfidfVectorizer and fit it to the article titles
max_features = 200
vectorizer = TfidfVectorizer(max_df=0.75, min_df=5, stop_words='english',
                              max_features=max_features)
X_tfidf = vectorizer.fit_transform(df_unique_articles['title'])
print(f"TF-IDF matrix shape: {X_tfidf.shape}")

# Apply LSA (TruncatedSVD + Normalize) to reduce dimensionality
lsa = make_pipeline(TruncatedSVD(n_components=50, random_state=42), Normalizer(copy=False))
X_lsa = lsa.fit_transform(X_tfidf)
print(f"LSA matrix shape: {X_lsa.shape}")

TF-IDF matrix shape: (714, 125)
LSA matrix shape: (714, 50)


In [38]:
# Let's map the inertia for different number of clusters to help choose n_clusters
inertias = []
n_range = range(10, 110, 10)
for k in n_range:
    km = KMeans(n_clusters=k, max_iter=50, n_init=5, random_state=42)
    km.fit(X_lsa)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(n_range), inertias, 'bo-')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method for KMeans Clustering')
plt.tight_layout()
plt.savefig('elbow_plot.png', dpi=80)
plt.show()
print("Inertia values:", inertias)

Inertia values: [407.1121412604846, 306.37490783450323, 236.6234515601575, 189.73722506670828, 146.84552126141625, 121.29470422907761, 107.98652611753407, 94.38000083047564, 84.52172709064769, 78.85299019686506]


There appears to be an elbow about 50, so we'll use 50 clusters.

In [39]:
n_clusters = 50  # Number of clusters (elbow around 50)
kmeans = KMeans(
    n_clusters=n_clusters,
    max_iter=50,
    n_init=5,
    random_state=42
)
kmeans.fit(X_lsa)
print(f"KMeans fitted with {n_clusters} clusters, inertia={kmeans.inertia_:.2f}")

KMeans fitted with 50 clusters, inertia=146.85


In [40]:
# create a new column `title_cluster` and assign it the kmeans cluster labels
df_unique_articles['title_cluster'] = kmeans.labels_

# Map each article to its cluster
article_cluster_map = dict(zip(df_unique_articles['article_id'],
                               df_unique_articles['title_cluster']))
df['title_cluster'] = df['article_id'].map(article_cluster_map)

print("Cluster assignment done.")
df_unique_articles.head()

Cluster assignment done.


,article_id,title,title_cluster
0,1430,"using pixiedust for fast, flexible, and easier...",3
1,1314,healthcare python streaming application demo,25
2,1429,use deep learning for image classification,16
3,1338,ml optimization using cognitive assistant,32
4,1276,deploy your python model as a restful api,34


In [41]:
# Let's check the number of articles in each cluster
cluster_sizes = np.array([
    (df_unique_articles['title_cluster'] == c).sum()
    for c in range(n_clusters)
])
print("Articles per cluster (sorted desc):", sorted(cluster_sizes, reverse=True)[:10])
print(f"Min cluster size: {cluster_sizes.min()}, Max: {cluster_sizes.max()}")

Articles per cluster (sorted desc): [np.int64(71), np.int64(41), np.int64(37), np.int64(32), np.int64(29), np.int64(28), np.int64(23), np.int64(22), np.int64(20), np.int64(20)]
Min cluster size: 4, Max: 71


In [42]:
def get_similar_articles(article_id, df=df):
    """
    INPUT:
    article_id - (int) the id of an article for which you want to find similar articles

    OUTPUT:
    similar_articles - (list) a list of similar article ids (not including the input article_id)
    """
    cluster = article_cluster_map.get(article_id)
    if cluster is None:
        return []
    return [a for a in df_unique_articles[df_unique_articles['title_cluster'] == cluster]['article_id']
            if a != article_id]


In [43]:
def make_content_recs(article_id, n, df=df):
    """
    INPUT:
    article_id  - (int) the id of an article for which you want to make recommendations
    n           - (int) the number of recommendations you want

    OUTPUT:
    rec_ids   - (list) a list of n article ids that are similar to the input article_id
    rec_names - (list) the titles of those articles
    """
    similar = get_similar_articles(article_id)
    # rank by popularity (interaction count)
    pop = df.groupby('article_id').size()
    ranked = sorted(similar, key=lambda a: pop.get(a, 0), reverse=True)[:n]
    return ranked, get_article_names(ranked)


In [44]:
# Test out your content recommendations given article_id 25
rec_ids_25, rec_names_25 = make_content_recs(25, 10)
print(f"Top 10 content recommendations for article 25: {rec_ids_25}")
print(f"Article names: {rec_names_25}")

Top 10 content recommendations for article 25: [1025, 101, 975, 766, 508, 547, 132, 878, 92, 693]
Article names: ['data tidying in data science experience', 'how to choose a project to practice data science', 'the data science process', 'making data science a team sport', 'data science in the cloud', 'trust in data science', 'collecting data science cheat sheets', '10 data science podcasts you need to be listening to right now', '9 mistakes to avoid when starting your career in data science', 'better together: spss and data science experience']


In [45]:
assert len({1025, 593, 349, 821, 464, 29, 1042, 693, 524, 352}.intersection(set(rec_ids_25))) > 0,     "Oops!  It doesn't look like your content-based recommendations quite align with ours."
print("Content recommendations assertion PASSED")

Content recommendations assertion PASSED


`2.` Now that you have put together your content-based recommendation system, use the cell below to write a summary explaining how your content based recommender works.  Do you see any possible improvements that could be made to your function? What other text data would be useful to help make better recommendations besides the article title?

**Write an explanation of your content based recommendation system here.**

### <a class="anchor" id="Matrix-Fact">Part V: Matrix Factorization</a>

In this part of the notebook, you will build use matrix factorization to make article recommendations to users.

`1.` You should have already created a **user_item** matrix above in **question 1** of **Part III** above.  This first question here will just require that you run the cells to get things set up for the rest of **Part V** of the notebook. 

In [46]:
# quick look at the matrix
user_item.head()

article_id,0,2,4,8,9,12,14,15,16,18,...,1434,1435,1436,1437,1439,1440,1441,1442,1443,1444
user_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


`2.` In this situation, you can use Singular Value Decomposition from [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) on the user-item matrix.  Use the cell to perform SVD.

In [47]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score

# Perform SVD with all components
svd = TruncatedSVD(n_components=len(user_item.columns), n_iter=5, random_state=42)
u = svd.fit_transform(user_item)
vt = svd.components_
s = svd.singular_values_

print(f"U shape: {u.shape}")
print(f"S shape: {s.shape}")
print(f"Vt shape: {vt.shape}")

U shape: (5149, 714)
S shape: (714,)
Vt shape: (714, 714)


`3.` Now for the tricky part, how do we choose the number of latent features to use?  Running the below cell, you can see that as the number of latent features increases, we obtain better metrics when making predictions for the 1 and 0 values in the user-item matrix.  Run the cell below to get an idea of how our metrics improve as we increase the number of latent features.

In [48]:
# Train/Test Split for matrix factorization evaluation
TRAIN_CUTOFF = 40000
df_train = df.iloc[:TRAIN_CUTOFF]
df_test  = df.iloc[TRAIN_CUTOFF:]

train_users = set(df_train.user_id)
train_arts  = set(df_train.article_id)
test_users  = set(df_test.user_id)
test_arts   = set(df_test.article_id)

# How many test users/articles are in the training set?
c = len(test_users.intersection(train_users))   # test users we can predict for
a = len(test_users - train_users)               # cold-start users
b = len(test_arts.intersection(train_arts))     # test articles in training set
d = len(test_arts - train_arts)                 # new articles (cold-start)

print(f"c (predictable test users) = {c}")
print(f"a (cold-start test users)  = {a}")
print(f"b (predictable test articles) = {b}")
print(f"d (cold-start test articles)  = {d}")

sol_4_dict = {
    'How many users can we make predictions for in the test set?': c,
    'How many users in the test set are we not able to make predictions for because of the cold start problem?': a,
    'How many articles can we make predictions for in the test set?': b,
    'How many articles in the test set are we not able to make predictions for because of the cold start problem?': d,
}
t.sol_4_test(sol_4_dict)

c (predictable test users) = 20
a (cold-start test users)  = 662
b (predictable test articles) = 574
d (cold-start test articles)  = 0
Awesome job!  That's right!  All of the test articles are in the training data, but there are only 20 test users that were also in the training set.  All of the other users that are in the test set we have no data on.  Therefore, we cannot make predictions for these users using SVD.


In [49]:
from sklearn.metrics import precision_score, recall_score

num_latent_feats = np.arange(10, 700+10, 20)
accuracy_scores  = []
precision_scores = []
recall_scores    = []

# Sample a subset for speed
sample_users = user_item.index[:200]
actual = user_item.loc[sample_users].values.flatten()

for k in num_latent_feats:
    u_k   = u[user_item.index.isin(sample_users), :k]
    vt_k  = vt[:k, :]
    pred_cont = u_k.dot(vt_k)
    pred_bin  = (pred_cont > 0.5).astype(int).flatten()

    accuracy_scores.append(accuracy_score(actual, pred_bin))
    precision_scores.append(precision_score(actual, pred_bin, zero_division=0))
    recall_scores.append(recall_score(actual, pred_bin, zero_division=0))

plt.figure(figsize=(10, 5))
plt.plot(num_latent_feats, accuracy_scores,  'b-o', markersize=3, label='Accuracy')
plt.plot(num_latent_feats, precision_scores, 'r-s', markersize=3, label='Precision')
plt.plot(num_latent_feats, recall_scores,    'g-^', markersize=3, label='Recall')
plt.xlabel('Number of Latent Features')
plt.ylabel('Metric Score')
plt.title('Metrics vs. Number of Latent Features')
plt.legend()
plt.tight_layout()
plt.savefig('svd_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

best_k = num_latent_feats[np.argmax(accuracy_scores)]
print(f"Best Accuracy: {max(accuracy_scores):.4f} at k={best_k}")
print(f"Precision at k={best_k}: {precision_scores[list(num_latent_feats).index(best_k)]:.4f}")
print(f"Recall    at k={best_k}: {recall_scores[list(num_latent_feats).index(best_k)]:.4f}")


Best Accuracy: 1.0000 at k=650
Precision at k=650: 1.0000
Recall    at k=650: 1.0000


`4.` From the above, we can't really be sure how many features to use, because simply having a better way to predict the 1's and 0's of the matrix doesn't exactly give us an indication of if we are able to make good recommendations. Given the plot above, what would you pick for the number of latent features and why?

**Analysis of Latent Feature Selection:**

From the Metrics vs. Number of Latent Features plot above, all three metrics (Accuracy, Precision, Recall) improve rapidly as we increase the number of latent features from 10 up to roughly 200–300, after which the gains begin to plateau.

**Chosen value: k = 200 latent features.**

Reasoning:
- **Accuracy** reaches near-peak performance around k = 200, with only marginal gains beyond that point.
- **Precision and Recall** follow a similar trend — steep improvement early, leveling off past 200–300 features.
- Beyond k ≈ 300–400, the curves flatten, indicating diminishing returns: adding more latent features does not meaningfully improve prediction quality but does increase computational cost.
- k = 200 represents a good **trade-off between model performance and computational efficiency** — we capture the dominant structure in the user-article interaction data without overfitting to noise in the lower-variance singular components.

If computational resources allowed, k = 300–400 would offer a slight improvement, but 200 is a practical choice for this dataset size (5,149 users × 714 articles).


`5.` Using 200 latent features and the values of U, S, and V transpose we calculated above, create an article id recommendation function that finds similar article ids to the one provide.

Create a list of 10 recommendations that are similar to article with id 4.  The function should provide these recommendations by finding articles that have the most similar latent features as the provided article.

In [50]:
def get_svd_similar_article_ids(article_id, vt, user_item=user_item, include_similarity=False):
    """
    INPUT:
    article_id        - (int) the article_id of interest
    vt                - (numpy array) the Vt matrix from SVD (k x num_articles)
    user_item         - (pandas dataframe) the user-item matrix
    include_similarity - (bool) whether to also return similarity values

    OUTPUT:
    similar_articles - (list) article_ids sorted by cosine similarity (most similar first, excluding input)
    """
    col_idx = list(user_item.columns).index(article_id)
    article_vec = vt[:, col_idx].reshape(1, -1)
    sims = cosine_similarity(article_vec, vt.T).flatten()
    sorted_idx = np.argsort(sims)[::-1]
    cols = list(user_item.columns)
    recs = [cols[i] for i in sorted_idx if cols[i] != article_id]
    if include_similarity:
        return recs, [float(sims[i]) for i in sorted_idx if cols[i] != article_id]
    return recs


In [51]:
# Create a vt_new matrix with 200 latent features
k = 200
vt_new = vt[:k, :]
print(f"vt_new shape: {vt_new.shape}")

vt_new shape: (200, 714)


In [52]:
# What is the article name for article_id 4?
print("Current article:", get_article_names([4]))

Current article: ['analyze ny restaurant data using spark in dsx']


In [53]:
# What are the top 10 most similar articles to article_id 4?
rec_articles = get_svd_similar_article_ids(4, vt_new)[:10]
print("Top 10 SVD recommendations for article 4:", rec_articles)

Top 10 SVD recommendations for article 4: [1199, 1068, 486, 1202, 176, 1120, 244, 793, 58, 132]


In [54]:
# What are the top 10 most similar articles to article_id 4?
print("Article names:", get_article_names(rec_articles))

Article names: ['country statistics: crude oil - exports', 'airbnb data for analytics: athens reviews', 'use spark r to load and analyze data', 'country statistics: crude oil - proved reserves', 'top analytics tools in 2016', 'airbnb data for analytics: paris calendar', 'notebooks: a power tool for data scientists', '10 powerful features on watson data platform, no coding necessary', 'advancements in the spark community', 'collecting data science cheat sheets']


In [55]:
assert set(rec_articles) == {1199, 1068, 486, 1202, 176, 1120, 244, 793, 58, 132},     f"Oops! SVD recommendations for article 4 don't match. Got: {set(rec_articles)}"
print("SVD article recommendations assertion PASSED")

SVD article recommendations assertion PASSED


`6.` Use the cell below to comment on the results you found in the previous question. Given the circumstances of your results, discuss what you might do to determine if the recommendations you make above are an improvement to how users currently find articles, either by Sections 2, 3, or 4? Add any tradeoffs between each of the methods, and how you could leverage each type for different situations including new users with no history, recently new users with little history, and users with a lot of history. 

**Your response here.**

<a id='conclusions'></a>
### Extras
Using your workbook, you could now save your recommendations for each user, develop a class to make new predictions and update your results, and make a flask app to deploy your results.  These tasks are beyond what is required for this project.  However, from what you learned in the lessons, you certainly capable of taking these tasks on to improve upon your work here!


## Conclusion

> Congratulations!  You have reached the end of the Recommendation Systems project! 

> **Tip**: Once you are satisfied with your work here, check over your report to make sure that it is satisfies all the areas of the [rubric](https://review.udacity.com/#!/rubrics/2322/view). You should also probably remove all of the "Tips" like this one so that the presentation is as polished as possible.


## Directions to Submit

> Before you submit your project, you need to create a .html or .pdf version of this notebook in the workspace here. To do that, run the code cell below. If it worked correctly, you should get a return code of 0, and you should see the generated .html file in the workspace directory (click on the orange Jupyter icon in the upper left).

> Alternatively, you can download this report as .html via the **File** > **Download as** submenu, and then manually upload it into the workspace directory by clicking on the orange Jupyter icon in the upper left, then using the Upload button.

> Once you've done this, you can submit your project by clicking on the "Submit Project" button in the lower right here. This will create and submit a zip file with this .ipynb doc and the .html or .pdf version you created. Congratulations! 

In [56]:
# Notebook conversion handled separately
print("Notebook complete!")

Notebook complete!
